In [1]:
!pip install captum torchvision scikit-image matplotlib pandas tqdm pyarrow seaborn


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 212.5 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.37.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which

In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [2]:
import os, gc, glob, torch, numpy as np, pandas as pd
from PIL import Image
from io import BytesIO
from tqdm import tqdm
import time
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from captum.attr import Lime
from skimage.segmentation import quickshift
import warnings
warnings.filterwarnings("ignore")

In [3]:
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

Device: cuda


In [4]:
DATASETS = {
    "cifar10": {
        "data_path": "/content/drive/MyDrive/xai-stability-benchmark/datasets/cifar_sample.npz",
        "result_dir": "/content/drive/MyDrive/xai-stability-benchmark/results_cifar10",
        "type": "npz",
        "num_classes": 1000,
    },
    "imagenet": {
        "data_path": "/content/drive/MyDrive/xai-stability-benchmark/datasets/imagenet_sample.parquet",
        "result_dir": "/content/drive/MyDrive/xai-stability-benchmark/results_imagenet",
        "type": "parquet",
        "num_classes": 1000,
    },
    "coco": {
        "data_path": "/content/drive/MyDrive/xai-stability-data/coco/train2017",
        "result_dir": "/content/drive/MyDrive/xai-stability-benchmark/results_coco",
        "type": "directory",
        "num_classes": 80,
    },
}
PERTURBATION_TYPES = ["rotation", "noise", "brightness", "translation", "jpeg"]
MODEL_NAMES = ["resnet50", "densenet121", "convnext_tiny", "vit_b_16"]

for cfg in DATASETS.values():
    os.makedirs(cfg["result_dir"], exist_ok=True)

print("Configuration loaded.")
print(f"Datasets:      {list(DATASETS.keys())}")
print(f"Perturbations: {PERTURBATION_TYPES}")
print(f"Models:        {MODEL_NAMES}")


Configuration loaded.
Datasets:      ['cifar10', 'imagenet', 'coco']
Perturbations: ['rotation', 'noise', 'brightness', 'translation', 'jpeg']
Models:        ['resnet50', 'densenet121', 'convnext_tiny', 'vit_b_16']


In [5]:
def get_ssim_spatial(attr1, attr2):
    def norm(x): return (x - x.min()) / (x.max() - x.min() + 1e-8)
    a1, a2 = norm(attr1), norm(attr2)
    pool = nn.AvgPool2d(11, stride=1, padding=5)
    mu1, mu2 = pool(a1), pool(a2)
    sigma12 = pool(a1 * a2) - mu1 * mu2
    sigma1_sq = pool(a1 * a1) - mu1 * mu1
    sigma2_sq = pool(a2 * a2) - mu2 * mu2
    c1, c2 = 0.01**2, 0.03**2
    ssim_map = ((2 * mu1 * mu2 + c1) * (2 * sigma12 + c2)) / \
               ((mu1**2 + mu2**2 + c1) * (sigma1_sq + sigma2_sq + c2))
    return ssim_map.view(attr1.shape[0], -1).mean(dim=1)


def get_spearman_rank(attr1, attr2):
    b = attr1.shape[0]
    v1, v2 = attr1.view(b, -1), attr2.view(b, -1)
    r1 = torch.argsort(torch.argsort(v1, dim=1), dim=1).float()
    r2 = torch.argsort(torch.argsort(v2, dim=1), dim=1).float()
    mu1, mu2 = r1.mean(dim=1, keepdim=True), r2.mean(dim=1, keepdim=True)
    num = ((r1 - mu1) * (r2 - mu2)).sum(dim=1)
    den = torch.sqrt(((r1 - mu1)**2).sum(dim=1) * ((r2 - mu2)**2).sum(dim=1)) + 1e-8
    return (num / den + 1) / 2


def get_top_k_jaccard(attr1, attr2, k=100):
    b = attr1.shape[0]
    v1, v2 = attr1.view(b, -1), attr2.view(b, -1)
    _, idx1 = torch.topk(v1, k, dim=1)
    _, idx2 = torch.topk(v2, k, dim=1)
    jaccards = []
    for i in range(b):
        mask = torch.isin(idx1[i], idx2[i])
        intersection = mask.sum().float()
        union = 2 * k - intersection
        jaccards.append(intersection / union)
    return torch.tensor(jaccards).to(DEVICE)


def calculate_fass(attr1, attr2):
    return (get_ssim_spatial(attr1, attr2) +
            get_spearman_rank(attr1, attr2) +
            get_top_k_jaccard(attr1, attr2)) / 3


print("FASS metrics defined.")


FASS metrics defined.


In [6]:
def apply_perturbation(img, p_type, to_tensor, norm_tf):
    if p_type == "rotation":
        pert = TF.rotate(img, 15)
        return norm_tf(to_tensor(pert))

    elif p_type == "noise":
        actual = to_tensor(img)
        noisy = torch.clamp(actual + torch.randn_like(actual) * 0.15, 0, 1)
        return norm_tf(noisy)

    elif p_type == "brightness":
        pert = TF.adjust_brightness(img, 1.5)
        return norm_tf(to_tensor(pert))

    elif p_type == "translation":
        pert = TF.affine(img, angle=0, translate=(20, 0), scale=1.0, shear=0)
        return norm_tf(to_tensor(pert))

    elif p_type == "jpeg":
        buf = BytesIO()
        img.save(buf, format="JPEG", quality=40)
        buf.seek(0)
        pert = Image.open(buf).convert("RGB")
        return norm_tf(to_tensor(pert))


print("Perturbation functions defined.")
print("rotation=15deg | noise=0.15 | brightness=1.5 | translation=20px | jpeg=q40")


Perturbation functions defined.
rotation=15deg | noise=0.15 | brightness=1.5 | translation=20px | jpeg=q40


In [7]:
class FASSCIFAR10Dataset(Dataset):
    def __init__(self, images, p_type):
        self.images = images
        self.p_type = p_type
        self.base_tf = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
        self.norm_tf = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.fromarray(self.images[idx]).convert("RGB")
        img = TF.resize(img, (224, 224))
        orig = self.base_tf(img)
        p = apply_perturbation(img, self.p_type, transforms.ToTensor(), self.norm_tf)
        return torch.stack([orig, p]), idx


class FASSImageNetDataset(Dataset):
    def __init__(self, dataframe, p_type):
        self.df = dataframe
        self.p_type = p_type
        self.base_tf = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
        self.norm_tf = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_data = row["image_bytes"]
        if isinstance(img_data, dict):
            img = Image.open(BytesIO(img_data["bytes"])).convert("RGB")
        elif isinstance(img_data, bytes):
            img = Image.open(BytesIO(img_data)).convert("RGB")
        else:
            img = img_data.convert("RGB")
        img = TF.resize(img, (224, 224))
        orig = self.base_tf(img)
        p = apply_perturbation(img, self.p_type, transforms.ToTensor(), self.norm_tf)
        return torch.stack([orig, p]), idx


class FASSCOCODataset(Dataset):
    def __init__(self, root, image_list, p_type):
        self.root = root
        self.image_list = image_list
        self.p_type = p_type
        self.base_tf = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
        self.norm_tf = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

    def __len__(self):
        return len(self.image_list)

    def __getitem__(self, idx):
        img = Image.open(os.path.join(self.root, self.image_list[idx])).convert("RGB")
        img = TF.resize(img, (224, 224))
        orig = self.base_tf(img)
        p = apply_perturbation(img, self.p_type, transforms.ToTensor(), self.norm_tf)
        return torch.stack([orig, p]), idx


print("Dataset classes defined (CIFAR-10 / ImageNet / COCO).")

Dataset classes defined (CIFAR-10 / ImageNet / COCO).


In [8]:
def prepare_models(num_classes):
    models_dict = {
        "resnet50": models.resnet50(weights="IMAGENET1K_V1"),
        "densenet121": models.densenet121(weights="IMAGENET1K_V1"),
        "convnext_tiny": models.convnext_tiny(weights="IMAGENET1K_V1"),
        "vit_b_16": models.vit_b_16(weights="IMAGENET1K_V1"),
    }

    if num_classes != 1000:
        for name, m in models_dict.items():
            if "resnet" in name:
                m.fc = nn.Linear(m.fc.in_features, num_classes)
            elif "dense" in name:
                m.classifier = nn.Linear(m.classifier.in_features, num_classes)
            elif "convnext" in name:
                m.classifier[2] = nn.Linear(m.classifier[2].in_features, num_classes)
            elif "vit" in name:
                m.heads.head = nn.Linear(m.heads.head.in_features, num_classes)

    for name, m in models_dict.items():
        for module in m.modules():
            if isinstance(module, nn.ReLU):
                module.inplace = False
        m.eval()

    return models_dict


def get_feature_mask(img_tensor):
    img_np = img_tensor.detach().cpu().numpy().transpose(1, 2, 0)
    img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)
    segments = quickshift(img_np, kernel_size=4, max_dist=200, ratio=0.2)
    return torch.tensor(segments).to(DEVICE)


print("Model preparation and feature mask functions defined.")

Model preparation and feature mask functions defined.


In [9]:
def load_dataset(ds_name):
    cfg = DATASETS[ds_name]

    if cfg["type"] == "npz":
        data = np.load(cfg["data_path"])
        for key in ["images", "x", "data"]:
            if key in data:
                arr = data[key]
                print(f"  [{ds_name}] Loaded {len(arr)} images from key '{key}', shape: {arr.shape}")
                return arr, len(arr)
        first_key = [k for k in data.keys() if data[k].ndim == 4][0]
        arr = data[first_key]
        print(f"  [{ds_name}] Loaded {len(arr)} images from key '{first_key}'")
        return arr, len(arr)

    elif cfg["type"] == "parquet":
        df = pd.read_parquet(cfg["data_path"])
        print(f"  [{ds_name}] Loaded {len(df)} images from parquet")
        return df, len(df)

    elif cfg["type"] == "directory":
        img_list = sorted([f for f in os.listdir(cfg["data_path"]) if f.endswith(".jpg")])[:40000]
        print(f"  [{ds_name}] Found {len(img_list)} images in directory")
        return img_list, len(img_list)


def make_dataloader(ds_name, data_obj, p_type, start_idx):
    cfg = DATASETS[ds_name]

    if cfg["type"] == "npz":
        ds = FASSCIFAR10Dataset(data_obj[start_idx:], p_type)
    elif cfg["type"] == "parquet":
        ds = FASSImageNetDataset(data_obj.iloc[start_idx:].reset_index(drop=True), p_type)
    elif cfg["type"] == "directory":
        ds = FASSCOCODataset(cfg["data_path"], data_obj[start_idx:], p_type)

    return DataLoader(ds, batch_size=64, num_workers=4, pin_memory=(DEVICE == "cuda"))


print("Data loading functions defined.")

Data loading functions defined.


In [10]:
def run_lime_single(model_name, model, ds_name, data_obj, total_images, p_type):
    result_dir = DATASETS[ds_name]["result_dir"]
    path = os.path.join(result_dir, f"{model_name}_fass_lime_{p_type}.csv")

    start = 0
    if os.path.exists(path):
        try:
            df = pd.read_csv(path)
            if not df.empty:
                start = df["image_index"].max() + 1
        except:
            pass
    if start >= total_images:
        print(f"      SKIP: already complete ({start} images).")
        return

    print(f"      Running from image {start}/{total_images}")

    loader = make_dataloader(ds_name, data_obj, p_type, start)
    lime = Lime(model)
    results = []
    start_time = time.time()
    processed = 0

    for stack, idxs in tqdm(loader, desc=f"{model_name}/{p_type}",
                            leave=False, mininterval=5.0):
        orig = stack[:, 0].to(DEVICE).requires_grad_(True)
        p = stack[:, 1].to(DEVICE)

        with torch.no_grad():
            pred_orig = torch.argmax(model(orig), 1)
            pred_pert = torch.argmax(model(p), 1)
            pred_match = (pred_orig == pred_pert)

        if not pred_match.any():
            processed += 1
            continue

        try:
            fm_orig = torch.stack([get_feature_mask(x) for x in orig[pred_match]])
            fm_pert = torch.stack([get_feature_mask(x) for x in p[pred_match]])

            attr_orig = lime.attribute(
                orig[pred_match], target=pred_orig[pred_match],
                n_samples=50, feature_mask=fm_orig,
            )

            p.requires_grad_(True)
            attr_pert = lime.attribute(
                p[pred_match], target=pred_pert[pred_match],
                n_samples=50, feature_mask=fm_pert,
            )

            res = calculate_fass(attr_orig, attr_pert)

            for i, val in enumerate(idxs[pred_match.cpu()]):
                results.append({
                    "image_index": val.item() + start,
                    "lime_fass": res[i].item(),
                })

        except Exception as e:
            if processed < 3:
                print(f"      Error on image {idxs[0].item() + start}: {e}")

        processed += 1
        torch.cuda.empty_cache()

        if len(results) >= 50:
            pd.DataFrame(results).to_csv(
                path, mode="a", header=not os.path.exists(path), index=False
            )
            elapsed = time.time() - start_time
            rate = processed / elapsed * 60 if elapsed > 0 else 0
            remaining = (total_images - start - processed)
            eta_h = (remaining / (rate / 60)) / 3600 if rate > 0 else 0
            print(f"      Checkpoint: {processed}/{total_images - start} | "
                  f"{rate:.0f} img/min | ETA: {eta_h:.1f}h")
            results = []

    if results:
        pd.DataFrame(results).to_csv(
            path, mode="a", header=not os.path.exists(path), index=False
        )

    elapsed = time.time() - start_time
    print(f"      Done in {elapsed/60:.1f} min")


print("Benchmark runner defined.")

Benchmark runner defined.


In [11]:
MAX_IMAGES_PER_DATASET = 2000  # cap for LIME (set to None for full dataset)

In [12]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

total_start = time.time()

for ds_name, cfg in DATASETS.items():

    print("\n" + "=" * 70)
    print(f"DATASET: {ds_name.upper()}")
    print(f"Source:  {cfg['data_path']}")
    print(f"Classes: {cfg['num_classes']}")
    print("=" * 70)

    # Check data exists
    if cfg["type"] == "directory" and not os.path.isdir(cfg["data_path"]):
        print(f"  Directory not found: {cfg['data_path']} -- skipping.")
        continue
    if cfg["type"] in ("npz", "parquet") and not os.path.isfile(cfg["data_path"]):
        print(f"  File not found: {cfg['data_path']} -- skipping.")
        continue

    # Load data once per dataset
    data_obj, total_images = load_dataset(ds_name)

# Cap dataset size for LIME
if MAX_IMAGES_PER_DATASET is not None:
    total_images = min(total_images, MAX_IMAGES_PER_DATASET)
    if cfg['type'] == 'npz':
        data_obj = data_obj[:total_images]
    elif cfg['type'] == 'parquet':
        data_obj = data_obj.iloc[:total_images].reset_index(drop=True)
    elif cfg['type'] == 'directory':
        data_obj = data_obj[:total_images]
    print(f'  Capped to {total_images} images')

    # Load models once per dataset (COCO needs 80-class heads)
    m_dict = prepare_models(cfg["num_classes"])

    for model_idx, model_name in enumerate(MODEL_NAMES, 1):
        model = m_dict[model_name]
        model.to(DEVICE)

        print(f"\n  [{model_idx}/4] {model_name}")

        for pert_idx, p_type in enumerate(PERTURBATION_TYPES, 1):
            print(f"    [{pert_idx}/5] {p_type}")
            run_lime_single(model_name, model, ds_name, data_obj, total_images, p_type)
            torch.cuda.empty_cache()

        model.cpu()
        del model
        gc.collect()
        torch.cuda.empty_cache()

    # Free dataset memory before next dataset
    del data_obj, m_dict
    gc.collect()
    torch.cuda.empty_cache()

total_elapsed = time.time() - total_start
print("\n" + "=" * 70)
print(f"ALL COMPLETE -- Total time: {total_elapsed/3600:.1f} hours")
print("=" * 70)



DATASET: CIFAR10
Source:  /content/drive/MyDrive/xai-stability-benchmark/datasets/cifar_sample.npz
Classes: 1000
  [cifar10] Loaded 10000 images from key 'images', shape: (10000, 3072)

DATASET: IMAGENET
Source:  /content/drive/MyDrive/xai-stability-benchmark/datasets/imagenet_sample.parquet
Classes: 1000
  [imagenet] Loaded 30000 images from parquet

DATASET: COCO
Source:  /content/drive/MyDrive/xai-stability-data/coco/train2017
Classes: 80
  [coco] Found 27217 images in directory
  Capped to 2000 images
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 315MB/s]


Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 148MB/s]


Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


...

      Done in 0.0 min
    [5/5] jpeg
      SKIP: already complete (2000 images).

  [3/4] convnext_tiny
    [1/5] rotation
      Running from image 1995/2000


      Done in 0.0 min
    [2/5] noise
      Running from image 1997/2000


      Done in 0.0 min
    [3/5] brightness
      SKIP: already complete (2000 images).
    [4/5] translation
      SKIP: already complete (2000 images).
    [5/5] jpeg
      Running from image 1997/2000


      Done in 0.0 min

  [4/4] vit_b_16
    [1/5] rotation
      SKIP: already complete (2000 images).
    [2/5] noise
      Running from image 1995/2000


      Done in 0.0 min
    [3/5] brightness
      SKIP: already complete (2000 images).
    [4/5] translation
      SKIP: already complete (2000 images).
    [5/5] jpeg
      SKIP: already complete (2000 images).

ALL COMPLETE -- Total time: 0.1 hours


In [13]:
print("\n" + "=" * 70)
print("RESULT VERIFICATION")
print("=" * 70)

grand_total = 0
grand_found = 0

for ds_name, cfg in DATASETS.items():
    result_dir = cfg["result_dir"]
    print(f"\n  {ds_name.upper()}:")

    for model_name in MODEL_NAMES:
        for p_type in PERTURBATION_TYPES:
            fname = f"{model_name}_fass_lime_{p_type}.csv"
            fpath = os.path.join(result_dir, fname)
            grand_total += 1
            if os.path.exists(fpath):
                df = pd.read_csv(fpath)
                valid = df["lime_fass"].notna().sum() if "lime_fass" in df.columns else 0
                print(f"    {fname:45s} {len(df):>6d} rows  ({valid} valid)")
                grand_found += 1
            else:
                print(f"    {fname:45s} MISSING")

print(f"\nFiles found: {grand_found}/{grand_total}")


RESULT VERIFICATION

  CIFAR10:
    resnet50_fass_lime_rotation.csv                 3720 rows  (3720 valid)
    resnet50_fass_lime_noise.csv                    2695 rows  (2695 valid)
    resnet50_fass_lime_brightness.csv               3718 rows  (3718 valid)
    resnet50_fass_lime_translation.csv              5661 rows  (5661 valid)
    resnet50_fass_lime_jpeg.csv                     9078 rows  (9078 valid)
    densenet121_fass_lime_rotation.csv              2073 rows  (2073 valid)
    densenet121_fass_lime_noise.csv                 1238 rows  (1238 valid)
    densenet121_fass_lime_brightness.csv            7082 rows  (7082 valid)
    densenet121_fass_lime_translation.csv           3411 rows  (3411 valid)
    densenet121_fass_lime_jpeg.csv                  9304 rows  (9304 valid)
    convnext_tiny_fass_lime_rotation.csv            4525 rows  (4525 valid)
    convnext_tiny_fass_lime_noise.csv               1169 rows  (1169 valid)
    convnext_tiny_fass_lime_brightness.csv          578

In [14]:
print("\n" + "=" * 70)
print("LIME FASS SUMMARY")
print("=" * 70)

for ds_name, cfg in DATASETS.items():
    result_dir = cfg["result_dir"]
    rows = []

    for model_name in MODEL_NAMES:
        for p_type in PERTURBATION_TYPES:
            fpath = os.path.join(result_dir, f"{model_name}_fass_lime_{p_type}.csv")
            if os.path.exists(fpath):
                df = pd.read_csv(fpath)
                if "lime_fass" in df.columns:
                    valid = df[df["lime_fass"].notna()]
                    if len(valid) > 0:
                        rows.append({
                            "model": model_name,
                            "perturbation": p_type,
                            "mean": valid["lime_fass"].mean(),
                            "std": valid["lime_fass"].std(),
                            "count": len(valid),
                        })

    if not rows:
        print(f"\n  {ds_name.upper()}: No results yet.")
        continue

    summary = pd.DataFrame(rows)
    summary.to_csv(os.path.join(result_dir, f"lime_summary_{ds_name}.csv"), index=False)

    # Per-model table
    print(f"\n  {ds_name.upper()}:")
    for model_name in MODEL_NAMES:
        mdf = summary[summary["model"] == model_name]
        if mdf.empty:
            continue
        print(f"\n    {model_name}:")
        for _, row in mdf.iterrows():
            print(f"      {row['perturbation']:15s}  "
                  f"FASS = {row['mean']:.4f} +/- {row['std']:.4f}  "
                  f"(n={int(row['count'])})")

    # LaTeX
    pivot = summary.pivot_table(values="mean", index="model", columns="perturbation")
    pivot_std = summary.pivot_table(values="std", index="model", columns="perturbation")

    print(f"\n  % LaTeX -- LIME FASS -- {ds_name.upper()}")
    print(r"  \begin{tabular}{l" + "c" * len(pivot.columns) + "}")
    print(r"  \toprule")
    print("  Model & " + " & ".join(pivot.columns) + r" \\")
    print(r"  \midrule")
    for model in pivot.index:
        vals = []
        for pert in pivot.columns:
            m = pivot.loc[model, pert]
            s = pivot_std.loc[model, pert]
            vals.append(f"{m:.3f} $\\pm$ {s:.3f}")
        print(f"  {model} & " + " & ".join(vals) + r" \\")
    print(r"  \bottomrule")
    print(r"  \end{tabular}")

print("\nDone.")









































LIME FASS SUMMARY

  CIFAR10:

    resnet50:
      rotation         FASS = 0.2823 +/- 0.0262  (n=3720)
      noise            FASS = 0.2841 +/- 0.0282  (n=2695)
      brightness       FASS = 0.3459 +/- 0.0479  (n=3718)
      translation      FASS = 0.3460 +/- 0.0457  (n=5661)
      jpeg             FASS = 0.3496 +/- 0.0481  (n=9078)

    densenet121:
      rotation         FASS = 0.2789 +/- 0.0265  (n=2073)
      noise            FASS = 0.2733 +/- 0.0293  (n=1238)
      brightness       FASS = 0.3379 +/- 0.0445  (n=7082)
      translation      FASS = 0.3387 +/- 0.0488  (n=3411)
      jpeg             FASS = 0.3441 +/- 0.0505  (n=9304)

    convnext_tiny:
      rotation         FASS = 0.2953 +/- 0.0265  (n=4525)
      noise            FASS = 0.2882 +/- 0.0289  (n=1169)
      brightness       FASS = 0.3561 +/- 0.0457  (n=5782)
      translation      FASS = 0.3620 +/- 0.0474  (n=9475)
      jpeg             FASS = 0.3635 +/- 0.0473  (n=9277)

    vit_b_16:
      rotation         FASS = 0